## NG-Notebook 3: Zero-shot classification with foundation models.

####
This notebook performs zero-shot classification on images to predict gaze classes given a raw image, image mask, and a label family. We test LAION CLIP and BiomedCLIP. This is the third notebook of the NeoGaze project, which aims to quantify physician visual attention during neonatal resuscitation.

We acknowledge the developers of (Open)CLIP and BiomedCLIP for their work. Please cite their papers if you use this code.

Contact: fparodi >> pennmedicine.upenn.edu

#### Set up environment

In [ ]:
!pip install transformers
!pip install datasets[vision]
!pip install accelerate

In [130]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from mobile_sam import sam_model_registry, SamPredictor
import open_clip
from transformers import pipeline
import torch

#### Load helper functions

In [89]:
# Helper functions:
RADIUS = 128

def get_greenest_pixel(image_array):
    green = image_array[:,:,1] - image_array.mean(axis=2)
    argmax2d_green = np.unravel_index(np.argmax(green), green.shape)
    return np.array([argmax2d_green[1], argmax2d_green[0]])

def subset_image_around_centroid(image_array, radius):
    greenest_point = get_greenest_pixel(image_array)
    y,x = greenest_point
    cropped_image = image_array[max(0, x - radius): min(image_array.shape[0], x + radius), 
                                max(0, y - radius): min(image_array.shape[1], y + radius)]
    return cropped_image

def load_and_preprocess_image(image_path):
    image = cv2.imread(image_path)
    input_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return input_image, image

def initialize_sam_model(model_type, checkpoint_path, device):
    sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
    sam.to(device=device)
    sam.eval()
    return sam

def segment_image(image_path, model_type="vit_t", checkpoint_path="mobile_sam.pt"):
    """
    Returns the segmented image.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load and initialize image and model:
    input_image, original_image = load_and_preprocess_image(image_path)
    sam = initialize_sam_model(model_type, checkpoint_path, device)
    predictor = SamPredictor(sam)
    
    # Use Tobii estimate as input point
    predictor.set_image(input_image)
    input_point = np.array([get_greenest_pixel(original_image)], dtype=float)
    input_label = np.array([1])

    # Perform prediction
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True,  # keep true to select the best mask
    )

    # Get best mask and create segmented image and masked portion:
    best_mask = np.argmax([np.sum(mask) for mask in masks])
    segmented_image_array = original_image * masks[best_mask].reshape(original_image.shape[0], original_image.shape[1], 1)

    coords = np.column_stack(np.where(masks[best_mask]))
    x_min, y_min = coords.min(axis=0)
    x_max, y_max = coords.max(axis=0)
    cropped_masked_portion = segmented_image_array[x_min:x_max+1, y_min:y_max+1]

    return Image.fromarray(segmented_image_array.astype('uint8')), Image.fromarray(cropped_masked_portion.astype('uint8'))

def load_biomed_clip(model_name="ViT-B-32-quickgelu", pretrained="laion400m_e32", cache_dir="biomed-clip-share", device="cpu"):
    model, _, preprocess_val = open_clip.create_model_and_transforms(model_name, pretrained, cache_dir)
    model.to(device).eval()
    tokenizer = open_clip.get_tokenizer(model_name)
    return model, preprocess_val, tokenizer

def biomed_clip_classify(image_inputs, model, preprocess_val, tokenizer, top_k=3, 
                    template='i am looking at ', labels=None, device='cpu'):
    metadata_list = []
    for img_input in image_inputs:
        # Check if the input is a file path or a PIL Image object
        if isinstance(img_input, str):
            img = Image.open(img_input)
            img_name = img_input.split('/')[-1]
        else:  # Assume it's a PIL Image object
            img = img_input
            img_name = "PIL_Image_Object"

        # Preprocess image and generate text
        img = preprocess_val(img).unsqueeze(0).to(device)
        text = tokenizer([template], context_length=77).to(device)

        # Get logits and convert to probabilities
        with torch.no_grad():
            image_features, text_features, logit_scale = model(img, text)
            logits = logit_scale * image_features @ text_features.t()
            logits = logits.softmax(dim=-1).cpu().numpy()
            sorted_indices = logits.argsort(axis=-1)[:, ::-1]

        # Get top k labels and probabilities
        top_probs = [f"{labels[idx]}: {logits[0][idx]*100:.1f}" for idx in sorted_indices[0, :top_k]]
        metadata = {'filename': img_name, 'top_probs': '\n'.join(top_probs)}
        metadata_list.append(metadata)

    return metadata_list

#### Test LAION OpenCLIP model

In [79]:
# openaiclip = "openai/clip-vit-large-patch14"
model = "laion/CLIP-ViT-B-14-DataComp.XL-s13B-b90K"
openaiclip_classifier = pipeline("zero-shot-image-classification", model=model)

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


In [117]:
# ======== Test one image =======

# Load image:
image_path = r'y:\neogaze\neogaze_deepethogram\DATA\Recording001 mf 2.10.22 from 00.00.00 to 00.02.18\frames\vid1_14_Vitals-Monitor.png'
image = cv2.imread(image_path)
original_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Crop image around Tobii estimate:
image_path_pil = Image.open(image_path)
image_array = np.array(image_path_pil)
cropped_image_array = subset_image_around_centroid(image_array, RADIUS)
cropped_image = Image.fromarray(cropped_image_array.astype('uint8'))

# Get mask around Tobii estimate:
segmented_image, masked_portion = segment_image(image_path)

# Display the original and cropped images side-by-side
fig, axes = plt.subplots(2, 2, figsize=(25, 25))
plt.rcParams.update({'font.size': 25})
fig.subplots_adjust(hspace=0.005, wspace=0.005)
axes[0, 0].imshow(Image.open(image_path)) # top-left
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')
axes[0, 1].imshow(cropped_image) # bottom-left
axes[0, 1].set_title('Cropped Image')
axes[0, 1].axis('off')
axes[1, 0].imshow(segmented_image) # bottom-right
axes[1, 0].set_title('Segmented Mask')
axes[1, 0].axis('off')
axes[1, 1].imshow(masked_portion)
axes[1, 1].set_title('Masked Portion')
axes[1, 1].axis('off')
plt.show()

<Figure size 2500x2500 with 4 Axes>

In [116]:
# print types of image_path, cropped_image, segmented_image, masked_portion
print(type(image_path))
print(type(cropped_image))
print(type(segmented_image))
print(type(masked_portion))

<class 'str'>
<class 'PIL.Image.Image'>
<class 'PIL.Image.Image'>
<class 'PIL.Image.Image'>


In [118]:
# Test LAION CLIP classifier on raw, cropped, and point-masked images:
labels_family = ['Infant',
          'Vitals Monitor',
          'CMAC Screen',
          'Airway Equipment',
          'Airway Provider',
          'Team Member',
          'Non-Team Member',
          'Other Physical Objects (e.g., wall)'
]
orig_score, crop_score, masked_score = openaiclip_classifier( # all pillow objects
    [image_path_pil, cropped_image, masked_portion],
    candidate_labels=labels_family,
)

print(f'Top 3 labels for original image: {[pred["label"] for pred in orig_score[:3]]}')
print(f'Top 3 labels for cropped image: {[pred["label"] for pred in crop_score[:3]]}')
print(f'Top 3 labels for masked image: {[pred["label"] for pred in masked_score[:3]]}')

Top 3 labels for original image: ['Airway Provider', 'Airway Equipment', 'Vitals Monitor']
Top 3 labels for cropped image: ['Vitals Monitor', 'CMAC Screen', 'Airway Equipment']
Top 3 labels for masked image: ['Other Physical Objects (e.g., wall)', 'Non-Team Member', 'Vitals Monitor']


In [132]:
# Now run OpenCLIP over directory of images, save top 3 labels for each frame to csv:

# Load image directory:
image_dir = r'Y:\neogaze\neogaze_deepethogram\DATA\Recording001 mf 2.10.22 from 00.00.00 to 00.02.18\frames'
labels_family = ['Infant',
          'Vitals Monitor',
          'CMAC Screen',
          'Airway Equipment',
          'Airway Provider',
          'Team Member',
          'Non-Team Member',
          'Other Physical Objects'
]
# Initialize an empty list to hold the data
data_list = []
df = pd.DataFrame(columns=['filename', 'top3_crop', 'top3_masked'])

for idx, image in enumerate(os.listdir(image_dir)):
    if not (image.endswith('.jpg') or image.endswith('.png')):
        continue
    image_path = os.path.join(image_dir, image)
    
    try:
        image_path_pil = Image.open(image_path)
        image_array = np.array(image_path_pil)
        cropped_image_array = subset_image_around_centroid(image_array, RADIUS)
        cropped_image = Image.fromarray(cropped_image_array.astype('uint8'))
        
        segmented_image, masked_portion = segment_image(image_path)
        
        crop_score, masked_score = openaiclip_classifier(
            [cropped_image, masked_portion],
            candidate_labels=labels_family,
        )
        
        top3_crop = [pred["label"] for pred in crop_score[:3]]
        top3_masked = [pred["label"] for pred in masked_score[:3]]
        
        data_list.append({
            'filename': image,
            'top3_crop': top3_crop,
            'top3_masked': top3_masked
        })
        if (idx + 1) % 500 == 0:
            print(f"{idx + 1}th image processed")
            
    except Exception as e:
        print(f"An error occurred while processing {image}: {e}")

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(data_list)

# Save the DataFrame to a CSV file
df.to_csv('video1_openclip_results.csv', index=False)

500th image processed
1000th image processed
1500th image processed
2000th image processed
2500th image processed


#### Load Microsoft BiomedCLIP model

In [129]:
# ======== Test one image =======
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, preprocess_val, tokenizer = load_open_clip(device=device)

score_dicts_list = biomed_clip_classify([image_path, cropped_image, masked_portion], model, preprocess_val,
                                        tokenizer, labels=labels_family, device=device)
print(score_dicts_list) # very bad.

[{'filename': 'y:\\neogaze\\neogaze_deepethogram\\DATA\\Recording001 mf 2.10.22 from 00.00.00 to 00.02.18\\frames\\vid1_14_Vitals-Monitor.png', 'top_probs': 'Infant: 100.0'}, {'filename': 'PIL_Image_Object', 'top_probs': 'Infant: 100.0'}, {'filename': 'PIL_Image_Object', 'top_probs': 'Infant: 100.0'}]
